# Dashboard Migration: Apply Permissions (Manual Workflow)

## Purpose

Apply permissions from source workspace to manually imported dashboards in target workspace.

## Prerequisites

1. ✅ Dashboards have been manually imported to target workspace (via UI, Bundle, or other method)
2. ✅ Permissions were captured during export (Notebook 02)
3. ✅ You know the target workspace URL and have appropriate credentials

## What This Notebook Does

1. Connects to target workspace
2. Lists dashboards in target location
3. Matches them with exported permissions by name/ID
4. Applies ACLs to each dashboard

**Note:** This works regardless of how dashboards were imported (UI, Bundle, SDK, etc.)

---

## Cell 1: Install Dependencies

In [ ]:
%pip install -U databricks-sdk pandas --quiet
dbutils.library.restartPython()

## Cell 2: Configuration

In [ ]:
# ============================================================================
# TARGET WORKSPACE (where dashboards were manually imported)
# ============================================================================

TARGET_WORKSPACE_URL = "https://fevm-akrishn-stable-classic-vv5y0k.cloud.databricks.com"
TARGET_PAT_TOKEN = dbutils.secrets.get(scope="migration", key="target-token")

# ============================================================================
# PATHS
# ============================================================================

VOLUME_BASE = "/Volumes/archana_krish_fe_dsa/vizient_deep_dive/dashboard_migration"
EXPORT_PATH = f"{VOLUME_BASE}/exported"  # Where permissions JSON files are

# ============================================================================
# TARGET DASHBOARD LOCATION
# ============================================================================

TARGET_PARENT_PATH = "/Shared/Migrated_Dashboards"  # Where dashboards were imported

# ============================================================================
# OPTIONS
# ============================================================================

DRY_RUN = True  # Set to False to actually apply permissions
MATCH_BY = "name"  # Options: "name", "id", "both"

print("✅ Configuration loaded")
print(f"   Target workspace: {TARGET_WORKSPACE_URL}")
print(f"   Looking for dashboards in: {TARGET_PARENT_PATH}")
print(f"   Permissions source: {EXPORT_PATH}")
print(f"   Dry run: {DRY_RUN}")

## Cell 3: Import Libraries

In [ ]:
import json
import pandas as pd
from typing import Dict, List, Optional
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.iam import PermissionLevel

print("✅ Libraries imported")

## Cell 4: Helper Functions

In [ ]:
def load_permissions_from_volume(export_path: str) -> Dict[str, Dict]:
    """Load all permissions JSON files from volume."""
    permissions_map = {}
    
    try:
        files = dbutils.fs.ls(export_path)
        perm_files = [f for f in files if '_permissions.json' in f.path]
        
        for perm_file in perm_files:
            content = dbutils.fs.head(perm_file.path, 10485760)
            perm_data = json.loads(content)
            
            dashboard_id = perm_data.get('dashboard_id')
            dashboard_name = perm_data.get('display_name')
            
            # Store by both ID and name for flexible matching
            if dashboard_id:
                permissions_map[dashboard_id] = perm_data
            if dashboard_name:
                permissions_map[dashboard_name] = perm_data
        
        return permissions_map
    except Exception as e:
        print(f"Error loading permissions: {e}")
        return {}

def list_target_dashboards(client: WorkspaceClient, parent_path: str) -> List[Dict]:
    """List dashboards in target workspace location."""
    dashboards = []
    
    for dash in client.lakeview.list():
        if dash.parent_path and dash.parent_path.startswith(parent_path):
            dashboards.append({
                'id': dash.dashboard_id,
                'name': dash.display_name,
                'path': dash.parent_path
            })
    
    return dashboards

def apply_permissions_to_dashboard(
    client: WorkspaceClient,
    dashboard_id: str,
    permissions_data: Dict,
    dry_run: bool = True
) -> Dict:
    """Apply permissions to a dashboard."""
    
    acl_list = permissions_data.get('access_control_list', [])
    
    if not acl_list:
        return {'status': 'skipped', 'reason': 'No permissions to apply'}
    
    if dry_run:
        return {
            'status': 'dry_run',
            'would_apply': len(acl_list),
            'permissions': acl_list
        }
    
    try:
        # Build ACL updates
        from databricks.sdk.service.iam import AccessControlRequest
        
        access_control_list = []
        for acl in acl_list:
            acr = AccessControlRequest()
            
            # Set principal
            if acl.get('user_name'):
                acr.user_name = acl['user_name']
            elif acl.get('group_name'):
                acr.group_name = acl['group_name']
            elif acl.get('service_principal_name'):
                acr.service_principal_name = acl['service_principal_name']
            else:
                continue  # Skip if no principal
            
            # Set permission level (use first/highest)
            if acl.get('all_permissions') and len(acl['all_permissions']) > 0:
                perm_str = acl['all_permissions'][0]
                # Convert string to PermissionLevel enum
                try:
                    acr.permission_level = PermissionLevel[perm_str] if hasattr(PermissionLevel, perm_str) else perm_str
                except:
                    acr.permission_level = perm_str
            else:
                continue  # Skip if no permission level
            
            access_control_list.append(acr)
        
        # Apply permissions
        client.permissions.update(
            request_object_type="dashboards",
            request_object_id=dashboard_id,
            access_control_list=access_control_list
        )
        
        return {
            'status': 'success',
            'applied': len(access_control_list)
        }
        
    except Exception as e:
        return {
            'status': 'failed',
            'error': str(e)
        }

print("✅ Helper functions loaded")

## Cell 5: Load Permissions and Discover Dashboards

In [ ]:
print("="*80)
print("LOADING PERMISSIONS AND DISCOVERING DASHBOARDS")
print("="*80)

# Connect to target workspace
target_client = WorkspaceClient(host=TARGET_WORKSPACE_URL, token=TARGET_PAT_TOKEN)
print(f"\n✅ Connected to target workspace")

# Load permissions from exported files
print(f"\n📋 Loading permissions from: {EXPORT_PATH}")
permissions_map = load_permissions_from_volume(EXPORT_PATH)
print(f"   Loaded permissions for {len(permissions_map)} dashboard(s)")

# List dashboards in target workspace
print(f"\n🔍 Discovering dashboards in: {TARGET_PARENT_PATH}")
target_dashboards = list_target_dashboards(target_client, TARGET_PARENT_PATH)
print(f"   Found {len(target_dashboards)} dashboard(s)")

# Display summary
if target_dashboards:
    df = pd.DataFrame(target_dashboards)
    display(df)

## Cell 6: Match and Apply Permissions

In [ ]:
print("\n" + "="*80)
print("MATCHING AND APPLYING PERMISSIONS")
print("="*80)

if not target_dashboards:
    print("\n⚠️  No dashboards found in target workspace!")
    print(f"   Check that dashboards were imported to: {TARGET_PARENT_PATH}")
elif not permissions_map:
    print("\n⚠️  No permissions found!")
    print(f"   Check that Notebook 02 captured permissions to: {EXPORT_PATH}")
else:
    results = []
    
    for dash in target_dashboards:
        dashboard_id = dash['id']
        dashboard_name = dash['name']
        
        print(f"\n📊 {dashboard_name}")
        print(f"   ID: {dashboard_id}")
        
        # Match permissions
        permissions_data = None
        matched_by = None
        
        if MATCH_BY in ["id", "both"]:
            # Try matching by ID
            permissions_data = permissions_map.get(dashboard_id)
            matched_by = "id"
        
        if not permissions_data and MATCH_BY in ["name", "both"]:
            # Try matching by name
            permissions_data = permissions_map.get(dashboard_name)
            matched_by = "name"
        
        if not permissions_data:
            print(f"   ⚠️  No matching permissions found")
            results.append({
                'dashboard_name': dashboard_name,
                'status': 'no_match',
                'matched_by': None
            })
            continue
        
        print(f"   ✅ Matched permissions by: {matched_by}")
        
        # Apply permissions
        result = apply_permissions_to_dashboard(
            target_client,
            dashboard_id,
            permissions_data,
            dry_run=DRY_RUN
        )
        
        if result['status'] == 'dry_run':
            print(f"   🔍 DRY RUN: Would apply {result['would_apply']} permission(s)")
            for perm in result['permissions']:
                principal = perm.get('user_name') or perm.get('group_name') or perm.get('service_principal_name')
                level = perm.get('all_permissions', ['unknown'])[0]
                print(f"      - {principal}: {level}")
        elif result['status'] == 'success':
            print(f"   ✅ Applied {result['applied']} permission(s)")
        elif result['status'] == 'failed':
            print(f"   ❌ Failed: {result['error']}")
        else:
            print(f"   ℹ️  {result.get('reason', 'Unknown status')}")
        
        results.append({
            'dashboard_name': dashboard_name,
            'dashboard_id': dashboard_id,
            'status': result['status'],
            'matched_by': matched_by,
            'details': result
        })
    
    # Summary
    print("\n" + "="*80)
    print("PERMISSION APPLICATION SUMMARY")
    print("="*80)
    
    summary_df = pd.DataFrame([{
        'Dashboard': r['dashboard_name'],
        'Status': r['status'],
        'Matched By': r['matched_by']
    } for r in results])
    
    display(summary_df)
    
    if DRY_RUN:
        print("\n⚠️  DRY RUN MODE - No permissions were actually applied")
        print("   Set DRY_RUN = False in Cell 2 to apply permissions")
    else:
        successful = len([r for r in results if r['status'] == 'success'])
        print(f"\n✅ Successfully applied permissions to {successful} dashboard(s)")